In [35]:
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

#### Данные и первичный анализ

In [2]:
dataset = load_dataset("emotion")
dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [19]:
# размеры сплитов
print("train:", len(dataset["train"]))
print("validation:", len(dataset["validation"]))
print("test:", len(dataset["test"]))

label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
label_names, num_labels


Train: 16000
Validation: 2000
Test: 2000


{'text': ['i didnt feel humiliated',
  'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake',
  'im grabbing a minute to post i feel greedy wrong',
  'i am ever feeling nostalgic about the fireplace i will know that it is still on the property',
  'i am feeling grouchy'],
 'label': [0, 0, 3, 2, 3]}

In [4]:
# 5 примеров
for i in range(5):
    ex = dataset["train"][i]
    print(f"TEXT: {ex['text']}")
    print(f"LABEL ID: {ex['label']}  ->  {label_names[ex['label']]}\n")



TEXT: i didnt feel humiliated
LABEL ID: 0  ->  sadness

TEXT: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
LABEL ID: 0  ->  sadness

TEXT: im grabbing a minute to post i feel greedy wrong
LABEL ID: 3  ->  anger

TEXT: i am ever feeling nostalgic about the fireplace i will know that it is still on the property
LABEL ID: 2  ->  love

TEXT: i am feeling grouchy
LABEL ID: 3  ->  anger



Датасет emotion подгрузился нормально, в нём три части: train, validation и test. Тексты короткие, в основном обычные фразы типа "I feel sad" или "I'm so happy today"  
6 классов с понятными названиями: sadness, joy, love, anger, fear, surprise
Анализировать тут буквально нечего, просто классификация эмоций по коротким предложениям


#### Токенизация

In [20]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

example_texts = [
    "I am very happy today!",
    "This is the worst day ever.",
    "I feel nothing."
]

for t in example_texts:
    tok = tokenizer(t, padding="max_length", truncation=True, max_length=16)
    print("\nTEXT:", t)
    print("TOKENS:", tokenizer.convert_ids_to_tokens(tok["input_ids"]))
    print("input_ids:", tok["input_ids"])
    print("attention_mask:", tok["attention_mask"])



TEXT: I am very happy today!
TOKENS: ['[CLS]', 'i', 'am', 'very', 'happy', 'today', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
input_ids: [101, 1045, 2572, 2200, 3407, 2651, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

TEXT: This is the worst day ever.
TOKENS: ['[CLS]', 'this', 'is', 'the', 'worst', 'day', 'ever', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
input_ids: [101, 2023, 2003, 1996, 5409, 2154, 2412, 1012, 102, 0, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]

TEXT: I feel nothing.
TOKENS: ['[CLS]', 'i', 'feel', 'nothing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
input_ids: [101, 1045, 2514, 2498, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


#### Инференс готовой модели

In [9]:
pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
).to(device)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 174.99it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def predict_with_model(model, texts):
    model.eval()
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)

    return preds.cpu().numpy(), probs.cpu().numpy()


In [11]:
sample_texts = [
    "I am so happy today!",
    "I feel really super down",
    "Bruh this is scary",
    "I love u like crazy",
    "I'm mad as hell rn"
]

pred_ids, pred_probs = predict_with_model(pretrained_model, sample_texts)

for text, pid, prob in zip(sample_texts, pred_ids, pred_probs):
    print("TEXT:", text)
    print("PRED_LABEL_ID:", int(pid), "->", label_names[int(pid)])
    print("CONFIDENCE:", float(prob[pid]), '\n')

TEXT: I am so happy today!
PRED_LABEL_ID: 3 -> anger
CONFIDENCE: 0.18784776329994202 

TEXT: I feel really super down
PRED_LABEL_ID: 3 -> anger
CONFIDENCE: 0.18647471070289612 

TEXT: Bruh this is scary
PRED_LABEL_ID: 3 -> anger
CONFIDENCE: 0.18732981383800507 

TEXT: I love u like crazy
PRED_LABEL_ID: 3 -> anger
CONFIDENCE: 0.18574266135692596 

TEXT: I'm mad as hell rn
PRED_LABEL_ID: 3 -> anger
CONFIDENCE: 0.18367259204387665 



Прогнала тексты через модель - она всё кидает в sadness, тк готовая модель вообще не различает эмоции (она не обучена на этот датасет)    
Она просто выбирает один и тот же класс всегда, скорее всего из-за случайных весов в классификаторе


#### Fine-tuning для классификации текста

In [23]:
max_length = 128

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )


tokenized_datasets = dataset.map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [24]:
num_labels = dataset["train"].features["label"].num_classes
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
).to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 175.96it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# метрики (accuracy + f1_macro)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}


In [27]:
# TrainingArguments + Trainer
output_dir = "artifacts/model_checkpoints"

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    seed=SEED,
    report_to="none"
)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [42]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.131748,0.197559,0.932500,0.906017


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]
c:\Users\ASUS Vivobook Flip S\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 

#### Оценка качества

In [32]:
test_metrics = trainer.evaluate(tokenized_datasets["test"])
test_metrics

c:\Users\ASUS Vivobook Flip S\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.1862991899251938,
 'eval_accuracy': 0.9235,
 'eval_f1_macro': 0.8746274466261362,
 'eval_runtime': 124.6461,
 'eval_samples_per_second': 16.045,
 'eval_steps_per_second': 1.003,
 'epoch': 3.0}

In [33]:
print("Test accuracy:", test_metrics["eval_accuracy"])
print("Test f1_macro:", test_metrics["eval_f1_macro"])

Test accuracy: 0.9235
Test f1_macro: 0.8746274466261362


In [36]:
# матрица ошибок
preds = trainer.predict(tokenized_datasets["test"])
y_true = preds.label_ids
y_pred = preds.predictions.argmax(axis=-1)

cm = confusion_matrix(y_true, y_pred)
labels = dataset["train"].features["label"].names

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()

plt.savefig("artifacts/confusion_matrix.png", dpi=200)
plt.close()


c:\Users\ASUS Vivobook Flip S\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [37]:
# Примеры предсказаний
texts = dataset["test"]["text"]
df = pd.DataFrame({
    "text": texts,
    "true_label": [labels[i] for i in y_true],
    "pred_label": [labels[i] for i in y_pred],
    "confidence": preds.predictions.max(axis=1)
})

df.head(20).to_csv("artifacts/sample_predictions.csv", index=False)
df.head(20)


,text,true_label,pred_label,confidence
0,im feeling rather rotten so im not very ambiti...,sadness,sadness,7.049687
1,im updating my blog because i feel shitty,sadness,sadness,7.106795
2,i never make her separate from me because i do...,sadness,sadness,7.032547
3,i left with my bouquet of red and yellow tulip...,joy,joy,6.890285
4,i was feeling a little vain when i did this one,sadness,sadness,7.008288
5,i cant walk into a shop anywhere where i do no...,fear,fear,6.383899
6,i felt anger when at the end of a telephone call,anger,anger,6.418897
7,i explain why i clung to a relationship with a...,joy,joy,3.703128
8,i like to have the same breathless feeling as ...,joy,joy,6.865667
9,i jest i feel grumpy tired and pre menstrual w...,anger,anger,6.854326


Модель в целом работает норм, accuracy и f1_macro получились примерно одинаковыми, значит по классам она не совсем разваливается  
На матрице ошибок видно, что лучше всего угадывает очевидные эмоции, а вот похожие типа fear/surprise и anger/sadness путает постоянно  
В примерах тоже видно, что короткие эмоциональные фразы она ловит, а вот нейтральные или двусмысленные - мимо